# Chapter 7 — What this actually was

We picked a problem with a known answer on purpose. Single-hand blackjack has an *exact* optimum — basic strategy, a lookup table proven optimal — so a learning agent could be measured against truth in every state. Six chapters used that ruler. This one steps back and asks what the whole exercise measured.

The short version was already on the table in Chapter 1, and nothing since has overturned it: **the table wins on exactness, simplicity, and zero tuning; the network is competent, not defeated.** What was *missing* in Chapter 1 was the why. We introduced the decision-tree frame there as a promise — three axes along which a lookup table and a neural network partition the same space differently — and said Chapter 7 would re-derive every finding from those three axes. That promise comes due here.

## 7.1 — The whole journey, one scoreboard

Every milestone, measured the same way — fresh shoe, 200k hands, agent and basic paired on the same seeded shoes, agreement as a back-half mean. The trimmed game (hit/stand/double) and the complete game (+split +surrender) side by side.

In [1]:
import sys; sys.path.insert(0, '.')
import statistics as st
import numpy as np, pandas as pd
from runs import load_runs, learning_curve, cell_categories, agreement_on, show, provenance, describe, OPTIMUM_PCT, EDGE_SE_PCT

df = load_runs(); dqn = df[df.method == 'dqn']
def back_half(path):
    a = [cp['agreement'] for cp in learning_curve(path) if cp.get('agreement') is not None]
    return st.mean(a[len(a)//2:]) if a else float('nan')

# --- trimmed game: the SAME selections Chapter 1 uses, so the synthesis cannot contradict it ---
net_grid = set(cell_categories(dqn[~dqn.with_splits].iloc[0].path))
tab   = df[(df.method=='tabular') & (df.episodes==5_000_000) & (~df.with_splits)]
naive = dqn[(dqn.hidden=='[64, 64]') & (dqn.episodes==1_000_000) & (~dqn.swa) & (dqn.lr_schedule=='constant')
            & (dqn.reward_baseline=='none') & (dqn.double_after==0) & (~dqn.with_splits)]
naive = naive[naive.edge_pct < 4.0]
best  = dqn[(~dqn.with_splits) & dqn.has_model].assign(bh=lambda d:[back_half(p) for p in d.path]).sort_values('bh').iloc[-1]
basic_a = df[(~df.with_splits) & df.basic_edge_pct.notna()].basic_edge_pct.mean()

# --- complete game (Chapter 6 selections) ---
tab_f  = df[df.with_splits & (df.method=='tabular')].sort_values('agreement').iloc[-1]
best_f = df[df.with_splits & (df.method=='dqn')].assign(bh=lambda d:[back_half(p) for p in d.path]).sort_values('bh').iloc[-1]
basic_f = df[df.with_splits & df.basic_edge_pct.notna()].basic_edge_pct.mean()

board = pd.DataFrame([
    {'game':'trimmed', 'method':'basic strategy (no-split optimum)', 'agreement':float('nan'), 'edge_pct':OPTIMUM_PCT['trimmed'], 'tuning':'—'},
    {'game':'trimmed', 'method':'tabular Q-learner',        'agreement':st.mean([agreement_on(p,net_grid) for p in tab.path]), 'edge_pct':tab.edge_pct.mean(), 'tuning':'none'},
    {'game':'trimmed', 'method':'naive DQN',                'agreement':naive.agreement.mean(), 'edge_pct':naive.edge_pct.mean(), 'tuning':'defaults'},
    {'game':'trimmed', 'method':'best DQN',                 'agreement':back_half(best.path), 'edge_pct':best.edge_pct, 'tuning':'full stack'},
    {'game':'+split','method':'basic strategy (full optimum)', 'agreement':float('nan'), 'edge_pct':OPTIMUM_PCT['complete'], 'tuning':'—'},
    {'game':'+split','method':'tabular Q-learner',        'agreement':back_half(tab_f.path) if not np.isnan(back_half(tab_f.path)) else tab_f.agreement, 'edge_pct':tab_f.edge_pct, 'tuning':'none'},
    {'game':'+split','method':'best DQN',                 'agreement':back_half(best_f.path), 'edge_pct':best_f.edge_pct, 'tuning':'full stack'},
])
show(board, pct=['agreement'], num=['edge_pct'],
     caption='The whole journey — agreement = match to the table on each game\'s grid (≈240 cells trimmed, ≈340–367 with splits); edge = %/hand vs basic, lower better. Tabular agreement is its converged value (no per-checkpoint log); DQN agreement is the back-half mean.',
     source='source: trimmed rows use Chapter 1\'s exact run selections; split rows use Chapter 6\'s (best expanded-game DQN is the 10M splits run; surrender is the 3-cell corner of Ch6.3, omitted here as no surrender-enabled tabular run exists). '
            'basic = mean over the no-split / split run pool respectively (fresh-shoe 200k, paired seed). '
            'tabular trimmed scored on the common 240-cell grid.')

game,method,agreement,edge_pct,tuning
trimmed,basic strategy (no-split optimum),nan%,1.11,—
trimmed,tabular Q-learner,92.8%,1.34,none
trimmed,naive DQN,82.1%,1.94,defaults
trimmed,best DQN,91.1%,0.90,full stack
+split,basic strategy (full optimum),nan%,0.54,—
+split,tabular Q-learner,94.0%,0.17,none
+split,best DQN,84.7%,0.74,full stack


**Reading it.** Two readings, the same as Chapter 1, now carried through to the full game.

On **agreement**, the table leads in both games and the lead *widens* as the game grows: a couple of points in the trimmed game (~0.93 vs ~0.91), about ten with splits added (~0.94 vs ~0.85). On **edge**, the trimmed-game learners are within noise of each other and of the no-split optimum (~1.11%): the gaps are a few tenths against a ±0.26%/hand eval band, so they can't be ranked. On the split game the table is clearly ahead (its ~0.17% vs the net's ~0.7%, a >2-SE gap). More actions, each with its own hard boundary (split here, surrender in Ch6.3),, cost the smooth approximator more; the exact table just stores them.

But hold the frame from Chapter 1: the per-game optima are ~1.11% (no-split) and ~0.54% (full); the trimmed learners land within noise of theirs, while on the full game the smooth net sits a few tenths above its 0.54% optimum. The story is not "the net failed." It is that the gap *has a shape* — and the shape is the same one axis, showing up wherever the game adds a wall.

## 7.2 — The three axes, earned

Chapter 1 said a network and a decision tree differ along three axes, and that every finding would be one of them. With the evidence in, here is the verdict on each.

| axis | what it *could* have caused | what the evidence actually showed |
|---|---|---|
| **boundary** — hard wall vs soft ramp | errors at every decision boundary | **the entire residual.** Over-doubling (Ch2, 4), over-splitting (Ch6), surrender-blindness (Ch6) are all one thing: a smooth function ramps where the table has a wall, so aggressive regions bleed outward and rare defensive islands never form. Encoding (Ch4) is the dial that buys back *some* wall, never all. |
| **cut direction** — axis-aligned vs oblique | the net mis-orienting the map | **a non-issue here.** Given an ordinal encoding the net laid the total/upcard axes out cleanly (Ch4.2 geometry) and carved coherent hit/stand/double regions. It represents the *orientation* fine; what it can't do is make the orientation's boundaries *sharp*. |
| **cells** — independent leaves vs shared weights | instability, and/or generalization | **both, and they cancel in our favour.** Shared weights coupled the double's value to its neighbours and made it *oscillate* (Ch2) — but that was curable with a decaying step (Ch3), pure dynamics, not a ceiling. The same sharing let the net answer *every* cell, including ones the table never visited — the generalization that gave it edge-parity in the trimmed game. |

So of the three axes, only the **boundary** axis is a genuine, permanent limit. The cut-direction axis was benign; the shared-cells axis caused a transient (fixable) wobble and bought a real advantage. The whole report collapses to one sentence: **a smooth approximator cannot place sharp, low-frequency boundaries — and optimal blackjack is made of them.** Everything else was either fixable (Ch3) or in the net's favour (generalization).

## 7.3 — Why we trust the shape

A finding this clean invites suspicion that it was manufactured. It was not, and Chapter 5 is the reason. Every number here is a *back-half mean*, never a transient peak; every comparison is *paired* (agent and basic on identical shoes) and *matched* (one variable at a time — the trap that nearly turned Chapter 3.3's lr-and-Double-DQN contamination into a false capacity effect); and we trusted only the metrics graded by the ruler — agreement and edge — over the ones the agent computes about itself, which shrink with confidence rather than correctness. The boundary residual survived all of that discipline. That is why it goes in as the conclusion and not a caveat.

## 7.4 — The reframe, and the bottom line

Here is the last turn of the frame Chapter 1 set up. On a problem whose model is *known*, computing the table is simply the right answer — exact, instant, untrainable-away. So the network's gap is not a failure to be closed; it is a **measurement** of something specific: the cost of solving the problem *model-free*, as if you did not know the rules. That cost, read off the scoreboard, is a couple of agreement points and near-zero edge in the trimmed game, widening to ~10 points and a few tenths of a percent once the full action set is on — and in exchange you get a policy that generalises to states no table ever visited. Stated that way, "the table wins" is not a defeat. It is the price tag on generality, and we now know its size and its shape: it is concentrated entirely on the sharp boundaries, and nowhere else.

**What this is honest about.** Several rows lean on single seeds; a matched [64,64] run confirms the trimmed-vs-complete gap widens ~2→10 points with the architecture held fixed (so the widening is real, not a net-size artifact), and that more capacity neither closed it nor reduced the over-splitting; evaluation is fresh-shoe throughout, which flatters basic's edge relative to a penetrated shoe; and splitting and surrender were a confirmation, not a study. None of these touch the central claim — the boundary residual appears in every configuration we ran — but they bound how hard any single number should be pushed.

**What it was, in the end.** A small, exactly-checkable RL problem, used not to build a blackjack bot but to ask *where a function approximator's policy departs from a known optimum, and why* — and to answer it with a single mechanism, three times confirmed, measured under a discipline strict enough to believe. The table wins. The network tells you, precisely, what generality costs.